# Preparation biomass carbon data

In [ ]:
# Libraries
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'

---

### Pre-processing

In [ ]:
# Get data
ds_esabio_agb = xr.open_zarr(f'{dir01}ds_prep_esabio_r001.zarr')
ds_spawn = xr.open_zarr(f'{dir01}ds_prep_spawn_r001.zarr')
ds_huang = xr.open_zarr(f'{dir01}ds_prep_huang_r001.zarr')

In [ ]:
# Calculate mean esabio carbon of 2015-2022 using IPCC default
da_esabio_agbc = ds_esabio_agb.to_array('a') \
    .mean('a') \
    .rename('esabio_agbc') * 0.47

# Merge esabio and spawn data into a single array
da_agbc = xr.merge([da_esabio_agbc, ds_spawn.spawn_biom_agbc]) \
    .to_array('agbc')

# Calculate agbc arrays with min, mean, and max values for each grid cell
#da_agbc_min = da_agbc.min('agbc')
#da_agbc_mean = da_agbc.mean('agbc')
da_agbc_max = da_agbc.max('agbc')

# Calculate bgbc arrays with min, mean, and max values for each grid cell
#da_bgbc_min = da_agbc_min * ds_huang.huang_bgbagbratio
#da_bgbc_mean = da_agbc_mean * ds_huang.huang_bgbagbratio
da_bgbc_max = da_agbc_max * ds_huang.huang_bgbagbratio

# Create cbiom (agbc + bgbc) array
da_cveg_biom = (da_agbc_max + da_bgbc_max).rename('cveg_biom')

# Create cveg_crop data
da_cveg_crop = (ds_spawn.spawn_crop_agbc + ds_spawn.spawn_crop_bgbc) \
    .rename('cveg_crop')

# Create cveg_gras data
da_cveg_gras = (ds_spawn.spawn_gras_agbc + ds_spawn.spawn_gras_bgbc) \
    .rename('cveg_gras')

# Create cveg_tund (for tundra from maximum values of biom and grassland)
da_cveg_tund = xr.merge([da_cveg_biom, da_cveg_gras]).to_array('a').max('a') \
    .rename('cveg_tund')

# Export arrays as netcdf for regridding
for da in [da_cveg_biom, da_cveg_crop, da_cveg_gras, da_cveg_tund]:
    # Attributes of latitude and longitude
    da.lat.attrs = dict(standard_name = 'latitude',
                            long_name = 'latitude',
                            units = 'degrees_north',
                            axis = 'Y')
    da.lon.attrs = dict(standard_name = 'longitude',
                            long_name = 'longitude',
                            units = 'degrees_east',
                            axis = 'X')
    # Export
    da.to_dataset() \
        .to_zarr(f'{dir02}ds_{da.name}_r001.zarr', mode='w');

In [ ]:
# Plot to check
ds = xr.open_mfdataset(f'{dir02}ds_cveg_*_r001.zarr', engine='zarr')

for i in range(0, len(ds.data_vars)):
    v = list(ds.data_vars)[i]
    fig, ax = plt.subplots(figsize=(10, 5), ncols=1, nrows=1)
    ds[v].plot.imshow(ax=ax, robust=True)
    ax.set_title(v)

---